In [2]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

In [3]:
model = genai.GenerativeModel("gemini-3.1-flash-lite")

response = model.generate_content("Recomiéndame en una frase qué visitar en San Sebastián")

print(response.text)

No puedes perderte un paseo por la **Bahía de La Concha** al atardecer seguido de una ruta de *pintxos* por los bares del **Parte Vieja**.


In [4]:
with open("docs/parte_vieja.md", "r", encoding="utf-8") as f:
    texto = f.read()

# Cortamos en cada "## " — cada corte es un monumento (o la intro, o fuentes)
secciones = texto.split("\n## ")
secciones = [s if s.startswith("#") else "## " + s for s in secciones]

# Excluimos la sección de Fuentes: son citas, no contenido que el bot deba usar para responder
chunks = [s.strip() for s in secciones if "## Fuentes" not in s]

print(f"Total de chunks: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"--- Chunk {i}: {c[:60]}...")

Total de chunks: 9
--- Chunk 0: # La Parte Vieja de San Sebastián (Donostia)

La Parte Vieja...
--- Chunk 1: ## Basílica de Santa María del Coro

La basílica de Santa Ma...
--- Chunk 2: ## Iglesia de San Vicente Mártir

La iglesia de San Vicente ...
--- Chunk 3: ## Plaza de la Constitución

La Plaza de la Constitución, co...
--- Chunk 4: ## Museo San Telmo

El Museo San Telmo, situado en la plaza ...
--- Chunk 5: ## Alameda del Boulevard

La Alameda del Boulevard, conocida...
--- Chunk 6: ## Calle 31 de Agosto

La calle 31 de Agosto es la única cal...
--- Chunk 7: ## Ayuntamiento de San Sebastián (antiguo Gran Casino)

El a...
--- Chunk 8: ## Monte Urgull (conjunto asociado a la Parte Vieja)

El mon...


In [5]:
resultado = genai.embed_content(
    model="models/gemini-embedding-001",
    content=chunks,
    task_type="retrieval_document",
    output_dimensionality=768
)

embeddings = resultado["embedding"]
print(f"Total de embeddings generados: {len(embeddings)}")
print(f"Dimensiones de cada vector: {len(embeddings[0])}")

Total de embeddings generados: 9
Dimensiones de cada vector: 768


In [6]:
import numpy as np

# Guardamos chunk + su vector juntos, para no perder la asociación
base_vectorial = list(zip(chunks, embeddings))

def buscar_relevantes(pregunta, top_n=2):
    # Convertimos la PREGUNTA a vector — nota el task_type distinto
    resultado_pregunta = genai.embed_content(
        model="models/gemini-embedding-001",
        content=pregunta,
        task_type="retrieval_query",
        output_dimensionality=768
    )
    vector_pregunta = resultado_pregunta["embedding"]

    # Comparamos ese vector contra los 9 vectores guardados
    similitudes = []
    for chunk_texto, chunk_vector in base_vectorial:
        similitud = np.dot(vector_pregunta, chunk_vector)
        similitudes.append((similitud, chunk_texto))

    # Ordenamos de más similar a menos, devolvemos los top_n
    similitudes.sort(key=lambda x: x[0], reverse=True)
    return [texto for _, texto in similitudes[:top_n]]

In [7]:
resultados = buscar_relevantes("¿cuándo se construyó la Basílica de Santa María?")
for r in resultados:
    print(r[:100], "\n---")

## Basílica de Santa María del Coro

La basílica de Santa María del Coro se encuentra en el extremo  
---
## Iglesia de San Vicente Mártir

La iglesia de San Vicente Mártir, en la calle San Juan, es conside 
---


In [8]:
def responder(pregunta, top_n=2):
    # Fase de consulta: buscamos los chunks relevantes (lo que ya probamos)
    chunks_relevantes = buscar_relevantes(pregunta, top_n=top_n)
    contexto = "\n\n---\n\n".join(chunks_relevantes)

    modelo_chat = genai.GenerativeModel(
        "gemini-3.1-flash-lite",
        system_instruction=(
            "Eres un guía turístico de San Sebastián. Responde SOLO con base en el "
            "contexto proporcionado. Si el contexto no tiene la respuesta, dilo "
            "claramente en vez de inventar información."
        )
    )

    prompt = f"Contexto:\n{contexto}\n\nPregunta del usuario: {pregunta}"
    respuesta = modelo_chat.generate_content(prompt)
    return respuesta.text

In [9]:
chunks_relevantes = buscar_relevantes("¿cuándo se construyó la Basílica de Santa María?")
contexto = "\n\n---\n\n".join(chunks_relevantes)

modelo_chat = genai.GenerativeModel(
    "gemini-3.1-flash-lite",
    system_instruction="Eres un guía turístico de San Sebastián. Responde SOLO con base en el contexto proporcionado."
)

prompt = f"Contexto:\n{contexto}\n\nPregunta del usuario: ¿cuándo se construyó la Basílica de Santa María?"
respuesta = modelo_chat.generate_content(prompt)

print(respuesta)

response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "La Bas\u00edlica de Santa Mar\u00eda del Coro ha tenido varias etapas de construcci\u00f3n:\n\n*   **Primer templo:** Entre los siglos XII y XIII existi\u00f3 una iglesia rom\u00e1nica.\n*   **Ampliaci\u00f3n:** Entre 1522 y 1560 se ampli\u00f3 en estilo g\u00f3tico-renacentista.\n*   **Reconstrucci\u00f3n barroca:** Tras los da\u00f1os sufridos en 1688, la reconstrucci\u00f3n en estilo barroco comenz\u00f3 en 1743 y concluy\u00f3 en 1774."
              }
            ],
            "role": "model"
          },
          "finish_reason": "STOP",
          "index": 0
        }
      ],
      "usage_metadata": {
        "prompt_token_count": 923,
        "candidates_token_count": 115,
        "total_token_count": 1038
      },
      "model_version": "gemini-3.

In [10]:
print(responder("¿Qué monumentos son imprecindibles en la parte vieja?"))

Como guía turístico, te informo que en la Parte Vieja de San Sebastián se concentra el patrimonio histórico y religioso más antiguo de la ciudad. Los elementos imprescindibles que puedes encontrar allí son:

*   **Las dos parroquias mayores.**
*   **Restos de la muralla.**
*   **El antiguo casco amurallado.**
*   **La calle 31 de Agosto**, la única calle que sobrevivió al incendio de 1813.

Además, aunque el monte Urgull está asociado a la Parte Vieja, en él podrás visitar el **Castillo de la Mota**, la **capilla del Santo Cristo de la Mota**, el **Cementerio de los Ingleses** y la **estatua monumental del Sagrado Corazón de Jesús**.


In [14]:
import requests
load_dotenv()


def consultar_clima(ciudad="San Sebastian,ES"):
    api_key = os.getenv("OPENWEATHER_API_KEY")
    url = "https://api.openweathermap.org/data/2.5/weather"
    parametros = {
        "q": ciudad,
        "appid": api_key,
        "units": "metric",
        "lang": "es"
    }

    respuesta = requests.get(url, params=parametros)

    if respuesta.status_code != 200:
        return f"No se pudo obtener el clima (código {respuesta.status_code})"

    datos = respuesta.json()
    descripcion = datos["weather"][0]["description"]
    temperatura = datos["main"]["temp"]
    sensacion = datos["main"]["feels_like"]

    return f"En San Sebastián: {descripcion}, {temperatura}°C (sensación {sensacion}°C)"

In [15]:
print(consultar_clima())


En San Sebastián: cielo claro, 22.19°C (sensación 22.74°C)


In [13]:
print(os.getenv("OPENWEATHER_API_KEY"))


None


In [16]:

tool_buscar_info = genai.protos.FunctionDeclaration(
    name="buscar_info_parte_vieja",
    description="Busca información histórica sobre monumentos y lugares de la Parte Vieja de San Sebastián: fechas de construcción, arquitectos, historia, estilo arquitectónico.",
    parameters={
        "type": "OBJECT",
        "properties": {
            "pregunta": {"type": "STRING", "description": "La pregunta específica sobre el lugar histórico"}
        },
        "required": ["pregunta"]
    }
)

tool_clima = genai.protos.FunctionDeclaration(
    name="consultar_clima",
    description="Obtiene el clima actual en tiempo real de San Sebastián: temperatura, condiciones.",
    parameters={"type": "OBJECT", "properties": {}}
)

herramientas = genai.protos.Tool(function_declarations=[tool_buscar_info, tool_clima])

In [17]:
def responder_agente(pregunta):
    modelo_agente = genai.GenerativeModel(
        "gemini-3.1-flash-lite",
        tools=[herramientas],
        system_instruction=(
            "Eres un guía turístico de San Sebastián especializado en la Parte Vieja. "
            "Usa las herramientas disponibles para responder. Si la pregunta no tiene "
            "relación con la Parte Vieja ni con el clima de San Sebastián, dilo "
            "claramente en vez de responder con conocimiento general."
        )
    )

    chat = modelo_agente.start_chat()
    respuesta = chat.send_message(pregunta)

    parte = respuesta.candidates[0].content.parts[0]

    # ¿El modelo pidió ejecutar una función, o ya respondió en texto?
    if not hasattr(parte, "function_call") or not parte.function_call.name:
        return respuesta.text  # Caso 3: ninguna tool aplicaba, respondió directo

    nombre_funcion = parte.function_call.name
    print(f"[DEBUG] El LLM eligió: {nombre_funcion}")  # para que veas la decisión mientras aprendes

    # Ejecutamos la función REAL correspondiente (tu código, no el LLM)
    if nombre_funcion == "buscar_info_parte_vieja":
        args = dict(parte.function_call.args)
        resultado = "\n\n".join(buscar_relevantes(args["pregunta"]))
    elif nombre_funcion == "consultar_clima":
        resultado = consultar_clima()
    else:
        resultado = "Función no reconocida."

    # Le devolvemos el resultado al modelo para que redacte la respuesta final
    respuesta_final = chat.send_message(
        genai.protos.Content(
            parts=[genai.protos.Part(
                function_response=genai.protos.FunctionResponse(
                    name=nombre_funcion,
                    response={"resultado": resultado}
                )
            )]
        )
    )
    return respuesta_final.text

In [18]:
print(responder_agente("¿cuándo se construyó la Basílica de Santa María?"))

[DEBUG] El LLM eligió: buscar_info_parte_vieja
La actual Basílica de Santa María del Coro, tal y como la vemos hoy en estilo barroco, se construyó entre **1743 y 1774**.

Es importante destacar que el edificio tiene una larga historia:
*   En ese mismo solar existió una iglesia románica desde los siglos XII-XIII.
*   Posteriormente, hubo una construcción gótico-renacentista entre 1522 y 1560.
*   Tras sufrir graves daños por una explosión en 1688, se llevó a cabo la reconstrucción barroca que comenzó en 1743 bajo la dirección de arquitectos como Ignacio de Ibero y concluyó en 1774.

Es uno de los monumentos más emblemáticos de la Parte Vieja, situada a los pies del monte Urgull.


In [23]:
def responder_agente_loop(pregunta, max_iteraciones=5):
    modelo_agente = genai.GenerativeModel(
        "gemini-3.1-flash-lite",
        tools=[herramientas],
        system_instruction=(
            "Eres un guía turístico de San Sebastián especializado en la Parte Vieja. "
            "Usa las herramientas disponibles para responder. Si la pregunta no tiene "
            "relación con la Parte Vieja ni con el clima de San Sebastián, dilo "
            "claramente en vez de responder con conocimiento general."
        )
    )

    chat = modelo_agente.start_chat()
    respuesta = chat.send_message(pregunta)

    for i in range(max_iteraciones):
        parte = respuesta.candidates[0].content.parts[0]

        # Si ya no pide función, terminó de decidir — devolvemos el texto
        if not hasattr(parte, "function_call") or not parte.function_call.name:
            return respuesta.text

        nombre_funcion = parte.function_call.name
        print(f"[DEBUG] Iteración {i+1}: el LLM eligió {nombre_funcion}")

        if nombre_funcion == "buscar_info_parte_vieja":
            args = dict(parte.function_call.args)
            resultado = "\n\n".join(buscar_relevantes(args["pregunta"]))
        elif nombre_funcion == "consultar_clima":
            resultado = consultar_clima()
        else:
            resultado = "Función no reconocida."

        # Mandamos el resultado y volvemos a preguntarle qué quiere hacer ahora
        respuesta = chat.send_message(
            genai.protos.Content(
                parts=[genai.protos.Part(
                    function_response=genai.protos.FunctionResponse(
                        name=nombre_funcion,
                        response={"resultado": resultado}
                    )
                )]
            )
        )

    return "Se alcanzó el límite de iteraciones sin llegar a una respuesta final."

In [20]:
print(responder_agente_loop("¿cuándo se construyó la Basílica de Santa María?"))

[DEBUG] Iteración 1: el LLM eligió buscar_info_parte_vieja
La construcción del edificio barroco que hoy conocemos como la Basílica de Santa María del Coro se llevó a cabo entre **1743 y 1774**.

Es importante destacar que su historia es más antigua:
*   En ese mismo solar existió previamente una iglesia románica construida entre los siglos XII y XIII.
*   Posteriormente, hubo una ampliación gótico-renacentista entre 1522 y 1560.
*   Tras los graves daños sufridos por una explosión en 1688, se inició la reconstrucción barroca actual en 1743, bajo la dirección de arquitectos como Pedro Ignacio de Lizardi, Miguel de Salazar e Ignacio de Ibero.


In [26]:
print(responder_agente_loop("¿qué tal está el clima hoy y qué me recomiendas visitar en la Parte Vieja?"))

[DEBUG] Iteración 1: el LLM eligió consultar_clima
¡Hola! Tienes un día fantástico en San Sebastián: el cielo está despejado y la temperatura es muy agradable, unos 22°C. Es un día perfecto para recorrer la Parte Vieja a pie.

Para aprovechar al máximo tu visita, aquí tienes mis recomendaciones imprescindibles en esta zona histórica:

1.  **Plaza de la Constitución:** Es el corazón de la Parte Vieja. Fíjate en los números de los balcones; antiguamente eran palcos para ver los espectáculos taurinos que se celebraban en la plaza.
2.  **Basílica de Santa María del Coro:** Una joya barroca impresionante. Te recomiendo mucho acercarte a ver su fachada principal y entrar para admirar su majestuoso interior.
3.  **Iglesia de San Vicente:** Es el edificio religioso más antiguo de la ciudad, de estilo gótico. Tiene un retablo mayor que es una auténtica maravilla.
4.  **Calle 31 de Agosto:** Es una de las calles con más historia y encanto, ya que fue la única que se salvó del incendio que destru

In [27]:
print(responder_agente_loop("Sí"))

¡Hola! Soy tu guía especializado en la Parte Vieja de San Sebastián. Estoy aquí para ayudarte a descubrir los secretos, la historia y la arquitectura de nuestro casco antiguo, o informarte sobre el clima actual de la ciudad.

¿En qué puedo ayudarte hoy? ¿Te gustaría saber más sobre algún edificio histórico, una plaza emblemática o quizás quieres saber qué tiempo hace antes de salir a pasear?
